<a href="https://colab.research.google.com/github/JSJeong-me/winnereye/blob/main/dgul/7_dgul_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install -r requirements.txt

# Data Load

In [1]:
from datetime import datetime
from pytz import timezone

# 현재 시간을 가져온다.
now = datetime.now(timezone('Asia/Seoul'))
print(now)

candidate = '윤석열'
portal = 'naver'
gen_date = '20220101'

# https://windybay.net/post/20/
date_time = now

2022-01-17 12:29:46.650904+09:00


In [ ]:
#import pymysql
from sqlalchemy import create_engine
import pandas.io.sql as pSql
import pandas as pd
import pickle

#my_list = ['a','b','c']
 
## Save pickle
#with open("data.pickle","wb") as fw:
#    pickle.dump(my_list, fw)
 
## Load pickle
with open("{0}.pkl".format(gen_date),"rb") as fr:
    data = pickle.load(fr)
#print(data)

df = pd.DataFrame(data)

df.head(5)

[데이터 엑셀로 저장]

# 2.2 데이터 전처리

In [3]:
df.loc[1]['rawdata']  # 전처리 전

'법정구속해야지 왜 안하는건지.. 못하는건지..'

[전처리]

In [4]:
import re

def remove_white_space(text):
    text = re.sub(r'[\t\r\n\f\v]', ' ', str(text))
    return text

def remove_special_char(text):
    text = re.sub('[^ ㄱ-ㅣ가-힣 0-9]+', ' ', str(text))
    return text

df.rawdata = df.rawdata.apply(remove_white_space)
df.rawdata = df.rawdata.apply(remove_special_char)


In [5]:
df.loc[1]['rawdata']  # 전처리 후

'법정구속해야지 왜 안하는건지  못하는건지 '

# 2.3 토크나이징 및 변수 생성

[토크나이징]

In [6]:
!pip install konlpy 

In [7]:
from konlpy.tag import Okt

okt = Okt()

df['rawdata_token'] = df.rawdata.apply(okt.morphs)
df['content_token'] = df.rawdata.apply(okt.nouns)

In [8]:
df['rawdata_token'].head(5)

0    [석, 열, 아, 일, 반, 국민, 들, 은, 잔고, 증명, 해서, 부을, 축척, ...
1                 [법정구속, 해야지, 왜, 안, 하는, 건지, 못, 하는, 건지]
2    [일반인, 이, 징역, 3년, 에, 1년, 추가, 인데, 보석, 으로, 풀려, 나온...
3    [못, 된거, 비리, 전, 부조, 사하여, 엄벌, 하, 라, 이, 것, 은, 시작,...
4    [잔고, 증명, 위조, 해서, 사기, 쳐도, 1년, 이내, 그것, 도, 구속, 안되...
Name: rawdata_token, dtype: object

In [9]:
df['content_token'].head(5)

0                       [열, 국민, 잔고, 증명, 축척, 몰수, 이, 공정]
1                                            [법정구속, 왜]
2    [일반인, 징역, 추가, 보석, 윤석열, 장모, 일이, 우리나라, 일반인, 나이, ...
3                                  [비리, 부조, 엄벌, 것, 시작]
4              [잔고, 증명, 위조, 사기, 그것, 구속, 이, 윤십원, 말, 공정]
Name: content_token, dtype: object

[파생변수 생성]

In [10]:
df['token_final'] = df.rawdata_token  + df.content_token

#df['count'] = df['count'].replace({',' : ''}, regex = True).apply(lambda x : int(x))

print(df.dtypes)

#df['label'] = df['count'].apply(lambda x: 'Yes' if x>=1000 else 'No')

nick             object
date             object
rawdata          object
clean_data       object
morphs           object
rawdata_token    object
content_token    object
token_final      object
dtype: object


In [11]:
df_drop = df['token_final'] # df[['token_final', 'label']]

In [12]:
df_drop.head()

0    [석, 열, 아, 일, 반, 국민, 들, 은, 잔고, 증명, 해서, 부을, 축척, ...
1        [법정구속, 해야지, 왜, 안, 하는, 건지, 못, 하는, 건지, 법정구속, 왜]
2    [일반인, 이, 징역, 3년, 에, 1년, 추가, 인데, 보석, 으로, 풀려, 나온...
3    [못, 된거, 비리, 전, 부조, 사하여, 엄벌, 하, 라, 이, 것, 은, 시작,...
4    [잔고, 증명, 위조, 해서, 사기, 쳐도, 1년, 이내, 그것, 도, 구속, 안되...
Name: token_final, dtype: object

[데이터 엑셀로 저장]

In [17]:
df_drop.to_csv('df_drop.csv', index = False, encoding = 'utf-8-sig')

# 2.4 단어 임베딩

[단어 임베딩]

In [ ]:
from gensim.models import Word2Vec

embedding_model = Word2Vec(df_drop, \
                           sg = 1, # skip-gram \
                           size = 100, \
                           window = 2, \
                           min_count = 1, \
                           workers = 4 \
                           )

print(embedding_model)

model_result = embedding_model.wv.most_similar("윤석열")
print(model_result)

[임베딩 모델 저장 및 로드]

In [26]:
from gensim.models import KeyedVectors

embedding_model.wv.save_word2vec_format('petitions_tokens_w2v') # 모델 저장
loaded_model = KeyedVectors.load_word2vec_format('petitions_tokens_w2v') # 모델 로드

model_result = loaded_model.most_similar("대통령")
print(model_result)

[('대통', 0.915793776512146), ('당선', 0.8583521842956543), ('이쯤', 0.8546522855758667), ('영부인', 0.8463578224182129), ('대권', 0.8166642189025879), ('흔들리겠다', 0.7936810255050659), ('검찰총장', 0.7753266096115112), ('대선', 0.7705039978027344), ('안되는거구나', 0.7703227400779724), ('반복', 0.7594432234764099)]


# 2.5 실험 설계

[데이터셋 분할 및 저장]

In [ ]:
from numpy.random import RandomState

rng = RandomState()

tr = df_drop.sample(frac=0.8, random_state=rng)
val = df_drop.loc[~df_drop.index.isin(tr.index)]

tr.to_csv('data/train.csv', index=False, encoding='utf-8-sig')
val.to_csv('data/validation.csv', index=False, encoding='utf-8-sig')

[Field클래스 정의]

In [ ]:
!!pip install -U torchtext==0.8.0

In [ ]:
#!pip install torch==1.6 torchtext==0.7

In [ ]:
!pip install https://github.com/pytorch/text/archive/master.zip

In [ ]:
!pip install -U torchtext==0.10.0

In [ ]:
import torchtext
from torchtext.legacy.data import Field

def tokenizer(text):
    text = re.sub('[\[\]\']', '', str(text))
    text = text.split(', ')
    return text

TEXT = Field(tokenize=tokenizer)
LABEL = Field(sequential = False)

[데이터 불러오기]

In [ ]:
from torchtext.legacy.data import TabularDataset

train, validation = TabularDataset.splits(
    path = 'data/',
    train = 'train.csv',
    validation = 'validation.csv',
    format = 'csv',
    fields = [('text', TEXT), ('label', LABEL)],
    skip_header = True
)

print("Train:", train[0].text,  train[0].label)
print("Validation:", validation[0].text, validation[0].label)

Train: ['에서', '일', '하고', '있는', '해외', '근로자', '들', '제발', '좀', '살려주세요', '저', '해외', '건설', '근로자', '하루하루', '일', '직원', '가족', '곳', '글', '처음', '글', '적지', '못', '관계자', '내', '가족', '일이', '생각', '한번', '대한민국', '청와대', '검색', '국민', '청원', '글자', '아래', '나라', '국민', '슬로건', '검색', '사람', '먼저', '문재인', '힘', '이란', '책', '제일', '먼저', '지금', '신랑', '생각', '벌렁', '거리', '가슴', '벌벌', '손', '꼭', '진짜', '절박', '심정', '글', '어디', '하소연', '여기', '글', '우리', '집', '가장', '위해', '내', '해', '줄', '수', '마지막', '일이', '생각', '마음', '저', '건설', '회사', '신랑', '결혼', '한지', '벌써', '직업', '특성', '회사', '발령', '이사도', '번', '말', '시간', '더', '딸', '지금', '중학교', '학년', '생후', '개월', '혼자', '지금', '두', '딸', '혼자', '결혼', '국내', '때', '회사', '일', '달', '번', '얼굴', '못', '벌써', '작년', '가을', '휴가', '때', '신랑', '건강', '문제', '해외', '말', '더', '그게', '제일', '후회', '성격', '고집', '세지', '워낙', '말', '책임감', '사람', '항상', '가정', '회사', '우선', '더', '이상', '휴가', '하루', '종일', '나', '자연인', '프로', '얼마나', '제', '구박', '지금', '곳', '일', '코로나', '담배', '쥐약', '차마', '담배', '좀', '말', '수', '그것', '낙', '요', '나라', '일', '둔부', '난', '종기', '하나', 

[단어장 및 DataLoader 정의]

In [ ]:
import torch
from torchtext.vocab import Vectors
from torchtext.legacy.data import BucketIterator

vectors = Vectors(name="data/petitions_tokens_w2v")

TEXT.build_vocab(train, vectors = vectors, min_freq = 1, max_size = None)
LABEL.build_vocab(train)

vocab = TEXT.vocab

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_iter, validation_iter = BucketIterator.splits(
    datasets = (train, validation),
    batch_size = 8,
    device = device,
    sort = False
)

print('임베딩 벡터의 개수와 차원 : {} '.format(TEXT.vocab.vectors.shape))

임베딩 벡터의 개수와 차원 : torch.Size([12483, 100]) 


# 2.6 TextCNN

[TextCNN 모델링]

In [ ]:
import torch.nn as nn   
import torch.optim as optim 
import torch.nn.functional as F 

class TextCNN(nn.Module): 
    
    def __init__(self, vocab_built, emb_dim, dim_channel, kernel_wins, num_class):
        
        super(TextCNN, self).__init__()
        
        self.embed = nn.Embedding(len(vocab_built), emb_dim)
        self.embed.weight.data.copy_(vocab_built.vectors)      
    
        self.convs = nn.ModuleList([nn.Conv2d(1, dim_channel, (w, emb_dim)) for w in kernel_wins])
        self.relu = nn.ReLU()                
        self.dropout = nn.Dropout(0.4)         
        self.fc = nn.Linear(len(kernel_wins)*dim_channel, num_class)     
        
    def forward(self, x):  
      
        emb_x = self.embed(x)           
        emb_x = emb_x.unsqueeze(1)  

        con_x = [self.relu(conv(emb_x)) for conv in self.convs]       

        pool_x = [F.max_pool1d(x.squeeze(-1), x.size()[2]) for x in con_x]    
        
        fc_x = torch.cat(pool_x, dim=1) 
        fc_x = fc_x.squeeze(-1)       
        fc_x = self.dropout(fc_x)         

        logit = self.fc(fc_x)     
        
        return logit

[모델 학습 함수 정의]

In [ ]:
def train(model, device, train_itr, optimizer):
    
    model.train()                               
    corrects, train_loss = 0.0,0        
    
    for batch in train_itr:
        
        text, target = batch.text, batch.label      
        text = torch.transpose(text, 0, 1)          
        target.data.sub_(1)                                 
        text, target = text.to(device), target.to(device)  

        optimizer.zero_grad()                           
        logit = model(text)                         
    
        loss = F.cross_entropy(logit, target)   
        loss.backward()  
        optimizer.step()  
        
        train_loss += loss.item()    
        result = torch.max(logit,1)[1] 
        corrects += (result.view(target.size()).data == target.data).sum()
        
    train_loss /= len(train_itr.dataset)
    accuracy = 100.0 * corrects / len(train_itr.dataset)

    return train_loss, accuracy

[모델 평가 함수 정의]

In [ ]:
def evaluate(model, device, itr):
    
    model.eval()
    corrects, test_loss = 0.0, 0

    for batch in itr:
        
        text = batch.text
        target = batch.label
        text = torch.transpose(text, 0, 1)
        target.data.sub_(1)
        text, target = text.to(device), target.to(device)
        
        logit = model(text)
        loss = F.cross_entropy(logit, target)

        test_loss += loss.item()
        result = torch.max(logit,1)[1]
        corrects += (result.view(target.size()).data == target.data).sum()

    test_loss /= len(itr.dataset) 
    accuracy = 100.0 * corrects / len(itr.dataset)
    
    return test_loss, accuracy

[모델 학습 및 성능 확인]

In [ ]:
model = TextCNN(vocab, 100, 10, [3, 4, 5], 2).to(device)
print(model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = optim.Adam(model.parameters(), lr=0.001)

best_test_acc = -1

for epoch in range(1, 3+1):
 
    tr_loss, tr_acc = train(model, device, train_iter, optimizer) 
    print('Train Epoch: {} \t Loss: {} \t Accuracy: {}%'.format(epoch, tr_loss, tr_acc))
    
    val_loss, val_acc = evaluate(model, device, validation_iter)
    print('Valid Epoch: {} \t Loss: {} \t Accuracy: {}%'.format(epoch, val_loss, val_acc))
        
    if val_acc > best_test_acc:
        best_test_acc = val_acc
        
        print("model saves at {} accuracy".format(best_test_acc))
        torch.save(model.state_dict(), "TextCNN_Best_Validation")
    
    print('-----------------------------------------------------------------------------')